In [1]:
# ============================================================
# MULTI-STEP SWE FORECASTING (1–30 DAYS)
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# PATHS
# ============================================================
DATA_PATH = '../../data_preprocessing/processed/shemonaikha_snow_dataset_2014_2023.csv'
FIG_DIR   = 'figures/'
TABLE_DIR = 'tables/'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print("=" * 60)
print("MULTI-STEP SWE FORECASTING (1–30 DAYS)")
print("=" * 60)

# ============================================================
# 1. Load and Prepare Data
# ============================================================
print("\n📁 STEP 1: DATA LOADING")
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)
df['swe'] = df['swe_final']   # target

# Fill missing values in numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')

print(f"Total days: {len(df)}")
print(f"Date range: {df['date'].min().date()} – {df['date'].max().date()}")

# ============================================================
# 2. Feature Engineering for Multi-Step Forecasting
# ============================================================
print("\n🔧 STEP 2: FEATURE ENGINEERING (LAGS, ROLLING, LEADS)")

df['snowfall'] = np.where(df['temperature_2m_C'] <= 0, df['precipitation_mm'], 0)
df['melt_potential'] = np.maximum(0, df['temperature_2m_C']) * 0.5

def create_multi_features(df, max_lag=30):
    df = df.copy()
    # Calendar features
    df['doy'] = df['date'].dt.dayofyear
    df['doy_sin'] = np.sin(2*np.pi*df['doy']/365)
    df['doy_cos'] = np.cos(2*np.pi*df['doy']/365)
    df['month'] = df['date'].dt.month
    df['month_sin'] = np.sin(2*np.pi*df['month']/12)
    df['month_cos'] = np.cos(2*np.pi*df['month']/12)
    
    # SWE lags (2..max_lag)
    for lag in range(2, max_lag+1):
        df[f'swe_lag{lag}'] = df['swe'].shift(lag)
    # SWE deltas
    for lag in range(2, max_lag):
        df[f'swe_delta{lag}'] = df[f'swe_lag{lag}'] - df[f'swe_lag{lag+1}']
    # Rolling statistics
    for w in [3,7,14,30]:
        df[f'swe_roll_mean{w}'] = df['swe'].shift(1).rolling(w, min_periods=1).mean()
        df[f'swe_roll_std{w}'] = df['swe'].shift(1).rolling(w, min_periods=1).std()
        df[f'swe_roll_max{w}'] = df['swe'].shift(1).rolling(w, min_periods=1).max()
        df[f'swe_roll_min{w}'] = df['swe'].shift(1).rolling(w, min_periods=1).min()
    # Meteorological lags
    met_vars = ['temperature_2m_C', 'precipitation_mm', 'snowfall', 'melt_potential',
                'snow_cover_ERA5', 'forecast_albedo', 'solar_radiation_MJ', 'thermal_radiation_MJ']
    for v in met_vars:
        if v in df.columns:
            for l in [1,3,7]:
                df[f'{v}_lag{l}'] = df[v].shift(l)
    # Targets (leads)
    for h in range(1, max_lag+1):
        df[f'swe_lead{h}'] = df['swe'].shift(-h)
    return df

df_feat = create_multi_features(df, max_lag=30)

# Select features and targets
feature_candidates = [c for c in df_feat.columns if 'lag' in c or 'roll' in c or 'delta' in c
                    or c.startswith('doy_') or c.startswith('month_')]
target_cols = [f'swe_lead{h}' for h in range(1,31)]
target_cols = [c for c in target_cols if c in df_feat.columns]

X = df_feat[feature_candidates]
y = df_feat[target_cols]
dates_all = df_feat['date']

# Remove rows with NaN in any target
mask = y.notna().all(axis=1)
X = X[mask].reset_index(drop=True)
y = y[mask].reset_index(drop=True)
dates_all = dates_all[mask].reset_index(drop=True)

print(f"Valid samples: {len(X)}")
print(f"Features: {len(feature_candidates)}")
print(f"Horizons: {len(target_cols)}")

# Train/test split (by date)
split_date = '2022-01-01'
train_mask = dates_all < split_date
test_mask = dates_all >= split_date
X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]
dates_train = dates_all[train_mask].reset_index(drop=True)
dates_test = dates_all[test_mask].reset_index(drop=True)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

# Imputation and scaling
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_imp)
X_test_s = scaler.transform(X_test_imp)

# ============================================================
# 3. Train Multi-Output Models
# ============================================================
print("\n🤖 STEP 3: MODEL TRAINING")

n_horizons = len(target_cols)
models = {}
# Random Forest
models['RF'] = MultiOutputRegressor(RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42), n_jobs=-1)
# XGBoost
models['XGB'] = MultiOutputRegressor(xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42), n_jobs=-1)
# CatBoost
models['Cat'] = MultiOutputRegressor(CatBoostRegressor(iterations=200, depth=6, learning_rate=0.05, verbose=False, random_seed=42), n_jobs=-1)

# Train each multi-output model
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_s, y_train)

# Stacking ensemble (per horizon)
print("Training stacking ensemble...")
stack_models = []
for h in range(n_horizons):
    base = [
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)),
        ('xgb', xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)),
        ('cat', CatBoostRegressor(iterations=100, depth=5, learning_rate=0.05, verbose=False, random_seed=42))
    ]
    stack = StackingRegressor(estimators=base, final_estimator=Ridge(), cv=3, n_jobs=-1)
    stack.fit(X_train_s, y_train.iloc[:, h])
    stack_models.append(stack)

# Predict
print("Predicting...")
preds = {}
for name, model in models.items():
    preds[name] = model.predict(X_test_s)
# Stacking prediction
preds['Stack'] = np.column_stack([m.predict(X_test_s) for m in stack_models])

# ============================================================
# 4. Metrics vs Horizon
# ============================================================
def calc_metrics_multi(y_true, y_pred):
    metrics = []
    for h in range(y_true.shape[1]):
        obs = y_true.iloc[:, h].values
        pre = y_pred[:, h]
        r2 = r2_score(obs, pre)
        mae = mean_absolute_error(obs, pre)
        rmse = np.sqrt(mean_squared_error(obs, pre))
        corr, _ = pearsonr(obs, pre)
        nse = 1 - np.sum((obs-pre)**2) / (np.sum((obs-np.mean(obs))**2) + 1e-6)
        metrics.append([h+1, r2, mae, rmse, nse, corr])
    return pd.DataFrame(metrics, columns=['horizon','R2','MAE','RMSE','NSE','Pearson_r'])

all_metrics = {}
for name, p in preds.items():
    all_metrics[name] = calc_metrics_multi(y_test, p)

# Print selected horizons
print("\nMulti-step forecast metrics (test period 2022–2023):")
print(f"{'Hor':<4} {'Model':<6} {'R²':<7} {'MAE':<6} {'RMSE':<6} {'NSE':<7}")
for h in [1,3,7,14,21,30]:
    for name in ['RF','XGB','Cat','Stack']:
        row = all_metrics[name].iloc[h-1]
        print(f"{h:2d}   {name:<6} {row['R2']:.3f}  {row['MAE']:5.1f}  {row['RMSE']:5.1f}  {row['NSE']:.3f}")
    print("-" * 50)

# Save metrics to tables/
metrics_concat = pd.concat(all_metrics.values(), keys=all_metrics.keys())
metrics_concat.to_csv(os.path.join(TABLE_DIR, 'multi_step_metrics.csv'))
print("✅ Metrics saved to tables/multi_step_metrics.csv")

# Build a DataFrame with dates and predictions for each horizon/model
forecast_rows = []
for i, date in enumerate(dates_test):
    row = {'date': date}
    for h in range(1, n_horizons+1):
        row[f'actual_h{h}'] = y_test.iloc[i, h-1]
        for name in preds:
            row[f'{name}_h{h}'] = preds[name][i, h-1]
    forecast_rows.append(row)

pd.DataFrame(forecast_rows).to_csv(os.path.join(TABLE_DIR, 'multi_step_predictions.csv'), index=False)
print("✅ Predictions saved to tables/multi_step_predictions.csv")

# ============================================================
# 5. Plots
# ============================================================
print("\n📈 Generating plots...")

# 5.1 Line plot: actual vs forecasts for different horizons (Ensemble)
selected_horizons = [1, 7, 21, 30]
colors_h = ['blue', 'green', 'orange', 'red']

fig, ax = plt.subplots(figsize=(14, 6))
# Actual SWE (using lead 1 as the observed series for dates_test + 1 day)
actual_dates = dates_test + pd.Timedelta(days=1)
ax.plot(actual_dates, y_test.iloc[:,0], 'k-', linewidth=2, label='Observed SWE')
for h, color in zip(selected_horizons, colors_h):
    pred_dates = dates_test + pd.Timedelta(days=h)
    ax.plot(pred_dates, preds['Stack'][:, h-1], color=color, linestyle='--', 
            label=f'Ensemble forecast H={h}d')
ax.set_xlabel('Date')
ax.set_ylabel('SWE (mm)')
ax.set_title('Multi-step forecasts for different horizons (Ensemble model)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'forecast_horizons_timeseries.png'), dpi=300)
plt.show()

# 5.2 Line plot: actual vs forecasts from all models for horizon 1
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(actual_dates, y_test.iloc[:,0], 'k-', linewidth=2, label='Observed')
for name, color in zip(['RF','XGB','Cat','Stack'], ['blue','green','purple','red']):
    ax.plot(actual_dates, preds[name][:,0], color=color, linestyle='--', label=name)
ax.set_xlabel('Date')
ax.set_ylabel('SWE (mm)')
ax.set_title('One-day-ahead forecasts from all models')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'forecast_1day_all_models.png'), dpi=300)
plt.show()

# 5.3 Degradation of R² and MAE with horizon
horizons = range(1,31)
fig, axes = plt.subplots(1,2, figsize=(14,5))
for name, color in zip(['RF','XGB','Cat','Stack'], ['blue','green','purple','red']):
    axes[0].plot(horizons, all_metrics[name]['R2'], '-', color=color, label=name)
    axes[1].plot(horizons, all_metrics[name]['MAE'], '-', color=color, label=name)
axes[0].set_xlabel('Forecast horizon (days)'); axes[0].set_ylabel('R²'); axes[0].legend(); axes[0].grid()
axes[1].set_xlabel('Forecast horizon (days)'); axes[1].set_ylabel('MAE (mm)'); axes[1].legend(); axes[1].grid()
plt.suptitle('Multi-Step SWE Forecast Performance')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'multistep_metrics.png'), dpi=300)
plt.show()

print("\n✅ Multi-step forecasting experiment completed.")

MULTI-STEP SWE FORECASTING (1–30 DAYS)

📁 STEP 1: DATA LOADING
Total days: 2410
Date range: 2014-01-01 – 2023-12-30

🔧 STEP 2: FEATURE ENGINEERING (LAGS, ROLLING, LEADS)
Valid samples: 2380
Features: 101
Horizons: 30
Train: 1937, Test: 443

🤖 STEP 3: MODEL TRAINING
Training RF...
Training XGB...
Training Cat...


CatBoostError: catboost/libs/train_lib/dir_helper.cpp:20: Can't create train working dir: catboost_info